<a href="https://colab.research.google.com/github/marcoastupp/ai-flux-training/blob/main/Treino_Flux_AI_Toolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import yaml
from google.colab import drive

# 1. Montar o Google Drive
drive.mount('/content/drive', force_remount=False)

# 2. Configurar o ambiente virtual Python 3.10
!add-apt-repository ppa:deadsnakes/ppa -y > /dev/null 2>&1
!apt-get update > /dev/null 2>&1
!apt-get install python3.10 python3.10-dev python3.10-venv python3.10-distutils -y > /dev/null 2>&1

!python3.10 -m venv /content/venv
import site
site.addsitedir('/content/venv/lib/python3.10/site-packages')

!/content/venv/bin/python -m pip install --upgrade pip > /dev/null 2>&1

# 3. Limpar e clonar o ai-toolkit travado num commit estável pré-Wan (setembro/2024)
%cd /content
!rm -rf /content/ai-toolkit
!git clone https://github.com/ostris/ai-toolkit.git
%cd /content/ai-toolkit
# Reverter para um commit super estável que não exige o WanTransformer3DModel
!git checkout 9212fd0 > /dev/null 2>&1 || git checkout main
!git submodule update --init --recursive

# Patch definitivo para o warn_once do torchao no PyTorch 2.3.1
import sys
import torch._dynamo.utils as dynamo_utils
if not hasattr(dynamo_utils, 'warn_once'):
    dynamo_utils.warn_once = lambda msg, *args, **kwargs: None

# 4. Limpar e instalar dependências com torchao compatível
print("📦 Limpando e instalando dependências essenciais...")
!/content/venv/bin/python -m pip uninstall -y torch torchvision torchaudio bitsandbytes opencv-python opencv-python-headless numpy albumentations av torchao > /dev/null 2>&1
!/content/venv/bin/python -m pip install -q --upgrade pip
!/content/venv/bin/python -m pip install -q numpy==1.26.4
!/content/venv/bin/python -m pip install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!/content/venv/bin/python -m pip install -q transformers==4.42.4 diffusers==0.29.2 accelerate==0.30.0
!/content/venv/bin/python -m pip install -q safetensors timm einops omegaconf sentencepiece peft bitsandbytes==0.43.1 python-dotenv oyaml opencv-python-headless==4.9.0.80 albumentations av
!/content/venv/bin/python -m pip install -q torchao

# Patch de compatibilidade para o warn_once do torchao no PyTorch 2.3.1
import sys
try:
    import torch._dynamo.utils as dynamo_utils
    if not hasattr(dynamo_utils, 'warn_once'):
        dynamo_utils.warn_once = lambda msg, *args, **kwargs: None
except Exception:
    pass

import torch._dynamo.utils as dynamo_utils
if not hasattr(dynamo_utils, 'warn_once'):
    dynamo_utils.warn_once = lambda msg, *args, **kwargs: None

# 5. Preparar diretórios e dataset
dataset_dir = "/content/dataset_max"
output_dir = "/content/drive/MyDrive/ComfyUI/models/loras"
img_train_dir = "/content/ai-toolkit/input_images"

os.makedirs(dataset_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
os.makedirs(img_train_dir, exist_ok=True)

print("📂 Sincronizando imagens e legendas...")
!cp -r /content/drive/MyDrive/ComfyUI/Max_dataset/* {dataset_dir}/
!rm -rf {img_train_dir}/*
os.makedirs(f"{img_train_dir}/10_sargento_max_wolf", exist_ok=True)
!cp -r {dataset_dir}/* {img_train_dir}/10_sargento_max_wolf/

# 6. Criar ficheiro de configuração YAML
config_content = """job: extension
config:
  name: sargento_max_wolf_flux_toolkit
  process:
    - type: train_lora
      training_folder: /content/ai-toolkit/output
      device: cuda:0
      trigger_word: "sargento max wolf"
      network:
        type: lora
        linear: 16
        linear_alpha: 16
      save:
        dtype: float16
        save_every: 250
        max_step_saves_to_keep: 3
        push_to_hub: false
      weighting_scheme: sigma_sqrt
      datasets:
        - folder_path: /content/ai-toolkit/input_images
          caption_ext: txt
          caption_dropout_rate: 0.05
          shuffle_tokens: false
          cache_latents_to_disk: true
          resolution: [512, 768]
      train:
        batch_size: 1
        steps: 600
        gradient_accumulation_steps: 4
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: flowmatch
        optimizer: adamw8bit
        lr: 1e-4
        ema_config:
          use_ema: false
        dtype: bf16
      model:
        name_or_path: black-forest-labs/FLUX.1-schnell
        is_flux: true
        quantize: false
"""

config_path = "/content/ai-toolkit/config_max.yaml"
with open(config_path, "w") as f:
    f.write(config_content)

print("🚀 A iniciar o treino do FLUX.1 [schnell] com a versão estável do ai-toolkit...")
!/content/venv/bin/python run.py /content/ai-toolkit/config_max.yaml

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
Cloning into 'ai-toolkit'...
remote: Enumerating objects: 15056, done.
remote: Counting objects: 100% (7222/7222), done.
remote: Compressing objects: 100% (1101/1101), done.
remote: Total 15056 (delta 6513), reused 6121 (delta 6121), pack-reused 7834 (from 3)
Receiving objects: 100% (15056/15056), 35.22 MiB | 23.14 MiB/s, done.
Resolving deltas: 100% (10442/10442), done.
/content/ai-toolkit
Already on 'main'
Your branch is up to date with 'origin/main'.
📦 Limpando e instalando dependências essenciais...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 0.30.0 requires torch>=1.10.0, which is not installed.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, which is not installed.
peft 0.21.2 requires torch>=1